# Unit 6 code companion: Inference Without a Formula

Get a ranking out of a forest, find out the default ranking is wrong, and build intervals for things no formula covers. Short cells, meant to be run one at a time.

Run the cells in order. Each section matches a moment in the slides. The point is to see the mechanism move when you change the inputs, so change them.

## Setup

The last column is added here and means nothing. That is the point of it.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(6)
rents = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/rent.csv")
rents = rents[(rents.Rent > 0) & (rents.Size > 0)].copy()
rents["id_number"] = rng.integers(100000, 999999, len(rents))
rents.head()

,BHK,Rent,Size,AreaType,City,FurnishingStatus,Preferred,Bathroom,Contact,id_number
0,2,10000,1100,Super Area,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner,500540
1,2,20000,800,Super Area,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner,584347
2,2,17000,1000,Super Area,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner,565985
3,2,10000,800,Super Area,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner,408943
4,2,7500,850,Carpet Area,Kolkata,Unfurnished,Bachelors,1,Contact Owner,951524


## 1. Fit a forest



In [2]:
cols = ["Size", "BHK", "Bathroom", "City", "FurnishingStatus", "id_number"]
X = pd.get_dummies(rents[cols], drop_first=True)
y = np.log(rents.Rent)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

forest = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1).fit(X_train, y_train)
print("forest, held-out R2:", round(forest.score(X_test, y_test), 3))

forest, held-out R2: 0.771


It predicts well. What it will not do is tell you which columns that came from, which is the rest of this notebook.

## 2. The importance the software gives you by default



In [3]:
impurity = pd.Series(forest.feature_importances_, index=X.columns)
impurity.sort_values(ascending=False).head(6).round(3)

Bathroom       0.288
City_Mumbai    0.279
Size           0.208
id_number      0.095
BHK            0.070
City_Delhi     0.019
dtype: float64

`id_number` is a column of random digits. Look at where it placed.

## 3. Permutation importance instead



In [4]:
perm = permutation_importance(forest, X_test, y_test, n_repeats=10,
                              random_state=0, n_jobs=-1)
permutation = pd.Series(perm.importances_mean, index=X.columns)
permutation.sort_values(ascending=False).head(6).round(3)

City_Mumbai                     0.699
Size                            0.350
Bathroom                        0.096
BHK                             0.038
City_Delhi                      0.031
FurnishingStatus_Unfurnished    0.019
dtype: float64

## 4. The two rankings side by side



In [5]:
pd.DataFrame({"impurity": impurity.round(3),
              "permutation": permutation.round(3)}).sort_values("impurity", ascending=False)

,impurity,permutation
Bathroom,0.288,0.096
City_Mumbai,0.279,0.699
Size,0.208,0.350
id_number,0.095,-0.002
BHK,0.070,0.038
City_Delhi,0.019,0.031
FurnishingStatus_Unfurnished,0.013,0.019
FurnishingStatus_Semi-Furnished,0.009,0.005
City_Kolkata,0.008,0.004
City_Hyderabad,0.007,0.013


The two disagree about the real columns and they disagree about the fake one. Only one of them puts a random number where it belongs.

## 5. One resample, to see what it is



In [6]:
one = rents.sample(len(rents), replace=True)
print("rows:", len(one), " distinct rows drawn:", one.index.nunique())
print("share of the original rows that appear at least once:",
      round(one.index.nunique() / len(rents), 3))

rows: 4743  distinct rows drawn: 3014
share of the original rows that appear at least once: 0.635


About 63% of the rows appear, some of them more than once. That reshuffling is the whole mechanism.

## 6. An interval where a formula already exists

Fit the elasticity 400 times, on 400 resamples.

In [7]:
RHS = "BHK + Bathroom + C(City) + C(FurnishingStatus)"

boot = []
for i in range(400):
    sample = rents.sample(len(rents), replace=True)
    fit = smf.ols(f"np.log(Rent) ~ np.log(Size) + {RHS}", data=sample).fit()
    boot.append(fit.params["np.log(Size)"])

print("bootstrap 95%:", np.percentile(boot, [2.5, 97.5]).round(4))

bootstrap 95%: [0.2366 0.2942]


## 7. What the formula says



In [8]:
full = smf.ols(f"np.log(Rent) ~ np.log(Size) + {RHS}", data=rents).fit()
print("estimate:", round(full.params["np.log(Size)"], 4))
print("formula 95%:", full.conf_int().loc["np.log(Size)"].values.round(4))

estimate: 0.2629
formula 95%: [0.2373 0.2886]


The two agree. That is the argument for trusting the method on quantities where no formula is available.

## 8. A gap between two medians, which has no formula



In [9]:
mumbai = rents.Rent[rents.City == "Mumbai"]
delhi = rents.Rent[rents.City == "Delhi"]

gap = [mumbai.sample(len(mumbai), replace=True).median()
       - delhi.sample(len(delhi), replace=True).median() for _ in range(2000)]

print("gap in medians:", f"Rs {mumbai.median() - delhi.median():,.0f}")
print("95% interval  :", [f"Rs {v:,.0f}" for v in np.percentile(gap, [2.5, 97.5])])

gap in medians: Rs 35,000
95% interval  : ['Rs 32,000', 'Rs 39,000']


Each city is resampled on its own, so each keeps its own sample size.

## 9. An interval on an importance score

Resample the training rows only. The test set stays as it is.

In [10]:
def importance_of(X_fit, y_fit, seed):
    f = RandomForestRegressor(n_estimators=40, random_state=0, n_jobs=-1).fit(X_fit, y_fit)
    r = permutation_importance(f, X_test, y_test, n_repeats=4, random_state=seed, n_jobs=-1)
    return pd.Series(r.importances_mean, index=X.columns)

rows = []
for i in range(25):
    take = rng.integers(0, len(X_train), len(X_train))
    rows.append(importance_of(X_train.iloc[take], y_train.iloc[take], i))
reps = pd.DataFrame(rows)

for c in ["City_Mumbai", "Size", "Bathroom", "BHK", "id_number"]:
    lo, hi = np.percentile(reps[c], [2.5, 97.5])
    print(f"{c:<14} [{lo:+.3f}, {hi:+.3f}]", "  straddles zero" if lo <= 0 <= hi else "")

City_Mumbai    [+0.644, +0.739] 
Size           [+0.251, +0.449] 
Bathroom       [+0.039, +0.193] 
BHK            [+0.019, +0.100] 
id_number      [-0.011, +0.006]   straddles zero


## 10. The mistake that makes everything look important

The same thing, resampling before the split instead of after.

In [11]:
rows = []
for i in range(25):
    sample = rents.sample(len(rents), replace=True)
    Xs = pd.get_dummies(sample[cols], drop_first=True)
    ys = np.log(sample.Rent)
    a, b, c, d = train_test_split(Xs, ys, test_size=0.3, random_state=i)
    f = RandomForestRegressor(n_estimators=40, random_state=0, n_jobs=-1).fit(a, c)
    r = permutation_importance(f, b, d, n_repeats=4, random_state=i, n_jobs=-1)
    rows.append(pd.Series(r.importances_mean, index=Xs.columns))

bad = pd.DataFrame(rows)
lo, hi = np.percentile(bad["id_number"], [2.5, 97.5])
print(f"id_number, resampling before the split: [{lo:+.3f}, {hi:+.3f}]")

id_number, resampling before the split: [+0.057, +0.085]


The same column of random digits now has an interval sitting well above zero. Resampling first puts the same listing in the training set and the test set, the forest remembers it, and shuffling any column appears to hurt. Nothing errors.

## 11. The whole method, as something you can reuse

Everything above is this function with a different third argument.

In [12]:
def bootstrap_interval(data, statistic, n=2000, level=95, seed=0):
    """Resample `data` with replacement, apply `statistic`, return the interval."""
    r = np.random.default_rng(seed)
    kept = [statistic(data.iloc[r.integers(0, len(data), len(data))]) for _ in range(n)]
    edge = (100 - level) / 2
    return np.percentile(kept, [edge, 100 - edge])


print("median rent      :", bootstrap_interval(rents, lambda s: s.Rent.median()).round(0))
print("mean rent        :", bootstrap_interval(rents, lambda s: s.Rent.mean()).round(0))
print("90th percentile  :", bootstrap_interval(rents, lambda s: s.Rent.quantile(0.9)).round(0))
print("share over 50k   :", bootstrap_interval(rents, lambda s: (s.Rent > 50000).mean()).round(4))

median rent      : [15000. 16000.]


mean rent        : [32910. 37336.]


90th percentile  : [68000. 75000.]


share over 50k   : [0.1417 0.1621]


Four quantities, four intervals, one function. Three of the four have no standard error formula you could have looked up.